# 第16章：认识已有电商系统

本篇依据[本章课件](https://github.com/xbsheng/atguigu-note/blob/bbe79814defe76ae597d07dd4f53f2e1fb26ed5d/agent_skills/%E8%AF%BE%E4%BB%B6/%E5%AE%9E%E6%88%98%E7%AF%87/02-%E8%AE%A4%E8%AF%86%E5%B7%B2%E6%9C%89%E7%94%B5%E5%95%86%E7%B3%BB%E7%BB%9F.md)整理，对应[视频 P27](https://www.bilibili.com/video/BV14rxM6qEnL?p=27)。资料快照：`bbe79814defe`（2026-10-29）。

以下命令、提示词和示例为课程阅读内容；样本数字是预期核对值，未在本次笔记整理中运行课程服务、安装技能或执行审批。

[完整配套工程](https://github.com/xbsheng/atguigu-note/tree/bbe79814defe76ae597d07dd4f53f2e1fb26ed5d/agent_skills/%E4%BB%A3%E7%A0%81/ecommerce-agent)；[业务口径](https://github.com/xbsheng/atguigu-note/blob/bbe79814defe76ae597d07dd4f53f2e1fb26ed5d/agent_skills/%E8%AF%BE%E4%BB%B6/%E5%AE%9E%E6%88%98%E7%AF%87/%E8%B5%84%E6%96%99/%E4%B8%9A%E5%8A%A1%E9%9C%80%E6%B1%82%E4%B8%8E%E6%95%B0%E6%8D%AE%E5%8F%A3%E5%BE%84.md)；[验收场景](https://github.com/xbsheng/atguigu-note/blob/bbe79814defe76ae597d07dd4f53f2e1fb26ed5d/agent_skills/%E8%AF%BE%E4%BB%B6/%E5%AE%9E%E6%88%98%E7%AF%87/%E8%B5%84%E6%96%99/%E9%AA%8C%E6%94%B6%E5%9C%BA%E6%99%AF%E4%B8%8E%E9%A2%84%E6%9C%9F%E7%BB%93%E6%9E%9C.md)。

## 16.1 从工作台观察能力

沿用第14章运行的8000服务，输入Agent令牌。异常订单页面负责查询/取证，工作流页面跟踪暂停恢复，工单页查看跟进记录，审批页供独立身份决策。初始无工单正常，建工单不等于发货。

保持固定支付日2026-10-10、基准2026-10-11 12:00 UTC+8，分页2条观察总数3和第二页A1003。当前操作为固定条件查询。

## 16.2 查询过滤与时间

过滤在business.py的scope、predicate、list_orders中执行，页面只传参和展示。条件含店铺、paid状态、付款日区间、付款早于as_of减24小时。数据库时间按UTC无时区datetime保存，接口转换为带+08:00时间，写读约定必须一致。

A1001/2/3分别等28/27/26小时；A1004仅18小时；已发货、取消和其他店铺不进入。总等待时间与超过阈值的时长分别表达。

### 用样例算一次严格边界

A1001在10月10日08:00支付，到11日12:00共28小时，超过阈值4小时；A1004从18:00到次日12:00仅18小时，不计。若付款时间恰好为10日12:00，则正好24小时，严格大于条件仍不满足。

支付日范围采用上海00:00至次日00:00，数据库统一UTC保存后，业务层先转换界限再比较。直接丢掉时区标识会使记录被分到错误自然日。

## 16.3 证据和规则分析

A1001需求2、现有0，提示库存风险；A1002现有20且有warehouse_hold，备注地址核验未通过；A1003现有20、未返回履约摘要，原因待确认。

order、inventory、fulfillment是查询事实，analysis是固定规则给出的可能原因和建议。没有记录不能直接认定仓库漏单，当前快照也不能恢复付款时的库存历史。

### 返回值里哪些是事实，哪些是解释

order中的状态和数量、inventory.available、fulfillment的event与note提供证据。analysis.possible_reason和suggestion来自现有程序的规则，是用于排查的解释。Agent应保留原来的确定程度，不能把字段名possible_reason翻成“已查明根因”。

A1002有地址核验未通过的拦截摘要，可以确认该摘要存在；其完整成因与解除时间仍要联系履约人员。A1003没返回摘要，只能继续核查下发与同步链路。

## 16.4 五张业务表与独立检查点

| 表 | 主要字段与约束 |
| --- | --- |
| orders | id主键、store、sku、quantity、status、paid_at |
| inventory | store+sku联合主键、available |
| fulfillment | order_id主键，event/note，每单一条最新摘要 |
| workflows | id、唯一request_id、store、params、decision、decided_at |
| tickets | id、唯一order_id、workflow_id、status |

SQLAlchemy Core用Table、select和事务连接访问。orders按门店+SKU找库存，按订单id找履约，处理流程和工单另存。seed.py仅初始化。LangGraph检查点另存节点状态，不能混同workflows登记表。

完整表定义保留在本篇后面的代码单元。

### 三类唯一约束解决三个问题

inventory的store+sku防止同门店商品有多条库存；workflows.request_id识别重复启动；tickets.order_id确保本例一个订单不会反复生成工单。它们作用范围不同，不能用一个工作流id替代所有业务幂等键。

fulfillment以order_id为主键，意味着保存每单一条最新摘要，不是完整事件日志。理解这个粒度，才能正确表达“没查到履约摘要”的边界。

## 16.5 页面对应HTTP接口

| 动作 | 接口 |
| --- | --- |
| 查超时订单 | GET /orders |
| 查证据 | GET /orders/{order_id}/evidence |
| 启动流程 | POST /workflows |
| 查流程 | GET /workflows/{workflow_id} |
| 人工决定 | POST /workflows/{workflow_id}/decision |
| 按决定恢复 | POST /workflows/{workflow_id}/resume |
| 查工单 | GET /tickets |

`/docs`查看接口。订单第一页的total=3、items两项、next_offset=2；后者为null才无下一页，不能将本页条数当总数。

Agent限店铺A，可查询推进；审批人可读流程并提交决定。Agent调用审批被拒绝，访问B1001证据返回404。页面文案不构成授权，真实检查在api.py。

## 16.6 三种已有程序分别验证

| 能力 | 输入 | 负责问题 |
| --- | --- | --- |
| checker.py | 本地JSON | 字段、类型、状态、时间结构 |
| business.py/api.py | 业务库 | 授权范围内超时订单和证据 |
| workflow.py | 查询条件和人工决定 | 保存清单、等待审批、恢复处理 |

samples/orders.json只有3条本地记录，数据库是7单，不能相互替代。valid=true仅代表格式通过，不代表发货或店铺授权。页面和MCP都复用已有程序，不再各自重写过滤逻辑。

## 16.7 单次查询链路

浏览器带Agent令牌与条件请求API → API检查身份参数并限定A → 业务函数固定查询数据库 → 返回total/items/next_offset → 页面展示与翻页。接入Agent后，模型负责选工具、组参数和解释，授权与查询仍在业务端。

### 页面成功后再加Agent入口

浏览器请求带Agent令牌，API先识别角色再把店铺A交给业务函数；业务函数按固定条件取数，返回总量和分页位置，页面只展示。后续MCP调用同一API，也会走相同限制。

若此时页面查询已失败，先修服务、身份或数据，不让模型用语言掩盖系统错误；若页面正常而MCP失败，再查连接和工具适配。逐层建立基线能缩小排查范围。

## 16.8 db.py完整表定义

库存联合主键、请求幂等键和订单工单唯一约束分别对应不同业务不变量。

[配套文件原文](https://github.com/xbsheng/atguigu-note/blob/bbe79814defe76ae597d07dd4f53f2e1fb26ed5d/agent_skills/%E4%BB%A3%E7%A0%81/ecommerce-agent/app/db.py)。代码按资料快照保留。

```python
"""业务表。时间统一以 UTC 无时区 datetime 存储。"""
from sqlalchemy import (MetaData, Table, Column, String, Integer, DateTime,
                        JSON, ForeignKey, create_engine)

metadata = MetaData()
orders = Table("orders", metadata,
    Column("id", String(32), primary_key=True),
    Column("store", String(16), nullable=False),
    Column("sku", String(32), nullable=False),
    Column("quantity", Integer, nullable=False),
    Column("status", String(16), nullable=False),
    Column("paid_at", DateTime, nullable=False))
inventory = Table("inventory", metadata,
    Column("store", String(16), primary_key=True),
    Column("sku", String(32), primary_key=True),
    Column("available", Integer, nullable=False))
fulfillment = Table("fulfillment", metadata,
    Column("order_id", String(32), ForeignKey("orders.id"), primary_key=True),
    Column("event", String(32), nullable=False),
    Column("note", String(200), nullable=False))
workflows = Table("workflows", metadata,
    Column("id", String(36), primary_key=True),
    Column("request_id", String(64), unique=True, nullable=False),
    Column("store", String(16), nullable=False),
    Column("params", JSON, nullable=False),
    Column("decision", String(16)),
    Column("decided_at", DateTime))
tickets = Table("tickets", metadata,
    Column("id", String(40), primary_key=True),
    Column("order_id", String(32), ForeignKey("orders.id"), unique=True, nullable=False),
    Column("workflow_id", String(36), ForeignKey("workflows.id"), nullable=False),
    Column("status", String(16), nullable=False))


def make_engine(url: str):
    kwargs = {"connect_args": {"check_same_thread": False}} if url.startswith("sqlite") else {}
    return create_engine(url, pool_pre_ping=True, **kwargs)
```